In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(patchwork)
library(RColorBrewer)

root <- "/Users/fupac/Desktop/DECODER单细胞/figure2"
rd <- file.path(root, "Affirmseq_rds")
STAMP <- readRDS(file.path(rd, "STAMP_annotated.rds"))
Threecell <- readRDS(file.path(rd, "Affirmseq_Threecell_annotated.rds"))
AffirmseqHM <- readRDS(file.path(rd, "AffirmseqHM_filtered_type_annotated.rds"))
merged_obj2 <- readRDS(file.path(rd, "AffirmseqHM_Stereocell_Harmony_merged_obj2.rds"))
Stereocell <- subset(readRDS("/Users/fupac/Downloads/cellline.merge.rds"), sample == "A03076G2")
if (is.factor(merged_obj2$dataset)) levels(merged_obj2$dataset) <- c("StereoCell", "Affirmseq-rep2", "Affirmseq-rep1")


In [ ]:
pal <- c(Affirmseq = "#D84040", STAMP = "#89AC46", Stereocell = "#377EB8")
pub <- theme_classic(base_size = 16) + theme(text = element_text(family = "Arial", color = "black"), axis.text = element_text(color = "black"), axis.title = element_text(color = "black"), legend.position = "none", plot.title = element_blank(), axis.title.x = element_blank())
hm <- AffirmseqHM@meta.data %>% select(human_counts, mouse_counts, type)
cnt <- table(hm$type)
pHM <- ggplot(hm, aes(human_counts, mouse_counts, color = type)) + geom_point(size = 2.5) + scale_color_manual(values = c(human = "#E41A1C", mouse = "#377EB8", undefine = "gray"), labels = c(paste0("human (", cnt["human"], ")"), paste0("mouse (", cnt["mouse"], ")"), paste0("mix (", cnt["undefine"], ")"))) + scale_x_continuous(limits = c(0, 50000), labels = function(x) paste0(x / 1000, "k")) + scale_y_continuous(limits = c(0, 50000), labels = function(x) paste0(x / 1000, "k")) + labs(x = "Human counts", y = "Mouse counts", color = "Cell Type") + theme_classic(base_size = 16) + theme(text = element_text(family = "Arial", color = "black"), legend.position = c(0.85, 0.85))
pThree <- DimPlot(Threecell, reduction = "umap", group.by = "cell_type", label = TRUE, pt.size = 2.5) + scale_color_brewer(palette = "Set1") + theme_classic() + theme(text = element_text(family = "Arial", size = 16, color = "black"), plot.title = element_blank(), axis.text = element_text(size = 16, color = "black"), axis.title = element_text(size = 16, color = "black")) + NoLegend()
df <- bind_rows(STAMP@meta.data %>% select(cell_type, nFeature_RNA, nCount_RNA) %>% mutate(tech = "STAMP"), Threecell@meta.data %>% select(cell_type, nFeature_RNA, nCount_RNA) %>% mutate(tech = "Affirmseq"), AffirmseqHM@meta.data %>% select(type, nFeature_RNA, nCount_RNA) %>% rename(cell_type = type) %>% mutate(tech = "Affirmseq"), Stereocell@meta.data %>% select(type, nFeature_RNA, nCount_RNA) %>% rename(cell_type = type) %>% mutate(tech = "Stereocell")) %>% filter(!tolower(cell_type) %in% c("undefined", "undefine"))
hm_df <- df %>% filter(cell_type %in% c("human", "mouse"), tech %in% c("Affirmseq", "Stereocell"))
ca_df <- df %>% filter(cell_type %in% c("SK-BR-3", "LNCaP", "MCF-7"), tech %in% c("Affirmseq", "STAMP"))
pHMbox <- ggplot(hm_df, aes(cell_type, nFeature_RNA, fill = tech)) + geom_boxplot(width = 0.8, position = position_dodge(0.9), outlier.shape = 21, outlier.size = 2) + scale_fill_manual(values = pal) + labs(y = "Gene count per cell") + pub
pCbox <- ggplot(ca_df, aes(cell_type, nFeature_RNA, fill = tech)) + geom_boxplot(width = 0.8, position = position_dodge(0.9), outlier.shape = 21, outlier.size = 2) + scale_fill_manual(values = pal) + labs(y = "Gene count per cell") + pub
p_main <- pHM + pThree + pHMbox + pCbox + plot_layout(widths = c(2, 2, 1.5, 2))
ggsave(file.path(root, "Affirmseq_Figure2_Gene.pdf"), p_main, width = 16, height = 4.5, device = cairo_pdf)
p_main


In [ ]:
Idents(merged_obj2) <- "dataset"
avg <- as.matrix(AverageExpression(merged_obj2, assays = "RNA", features = VariableFeatures(merged_obj2), layer = "data")$RNA)
xy <- as.data.frame(avg[, c("Affirmseq-rep1", "Affirmseq-rep2")])
xy <- log2(xy[rowSums(xy) > 0, ] + 1)
r <- cor(xy$`Affirmseq-rep1`, xy$`Affirmseq-rep2`, method = "pearson")
pUMAP <- DimPlot(merged_obj2, reduction = "umap", group.by = "dataset", split.by = "dataset", order = c("StereoCell", "Affirmseq-rep2", "Affirmseq-rep1"), pt.size = 0.1) + theme_classic() + theme(text = element_text(family = "Arial", size = 16), strip.text = element_text(family = "Arial", size = 16), axis.title = element_text(family = "Arial", size = 16), axis.text = element_text(family = "Arial", size = 16), legend.text = element_text(family = "Arial", size = 16), legend.title = element_text(family = "Arial", size = 16))
pCor <- ggplot(xy, aes(`Affirmseq-rep1`, `Affirmseq-rep2`)) + geom_point(size = 3, alpha = 0.4, color = "#377EB8") + geom_abline(intercept = 0, slope = 1, color = "#E41A1C", linetype = "dashed", linewidth = 1) + annotate("text", x = 0.5, y = max(xy) * 0.95, label = paste0("Pearson R = ", sprintf("%.4f", r)), family = "Arial", size = 6, hjust = 0) + labs(title = "Log2-Scale Correlation: Rep1 vs Rep2", x = "log2(Average Expression + 1) - Rep1", y = "log2(Average Expression + 1) - Rep2") + theme_classic() + theme(text = element_text(family = "Arial", size = 16), plot.title = element_text(hjust = 0.5, size = 18), axis.text = element_text(color = "black"), panel.border = element_rect(fill = NA, linewidth = 1.2))
p_rep <- pUMAP | pCor + plot_layout(widths = c(2, 1))
ggsave(file.path(root, "Affirmseq_Reproducibility.pdf"), p_rep, width = 16, height = 5, device = cairo_pdf)
p_rep
